In [1]:
import os
import sys
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import json

from pyperf.harness.utils import natural_sort_key
from pyperf.constants import SUBMISSIONS_DIR

In [2]:
# helpers
plots_dir = "plots"
os.makedirs(plots_dir, exist_ok=True)


def save_plot(fig, fname):
    fpath = os.path.join(plots_dir, fname)
    fig.savefig(fpath, dpi=300, bbox_inches="tight")
    print(f"Saved plot to {fpath}")
    plt.close(fig)


def get_traces(pattern: str):
    traces = list(SUBMISSIONS_DIR.glob(pattern))
    if traces == []:
        print(f"Could not find any traces matching pattern: {pattern}")
    traces = sorted([str(t) for t in traces], key=natural_sort_key)
    return [Path(t) for t in traces]


def load_json(fname):
    with open(fname, "r") as f:
        return json.load(f)

In [3]:
HOME_DIR = Path(os.path.expanduser("~"))
EVAL_REPORTS_DIR = HOME_DIR / "pyperf/reports/beat_k_reports"
eval_report = EVAL_REPORTS_DIR / "o3-mini.beat@10.test.report.json"
traces = get_traces(
    "manishs__pyperf-test/CodeActAgent/o3-mini_maxiter_50_N_v0.25.0-no-hint-run_*/output.jsonl"
)
submissions = get_traces(
    "manishs__pyperf-test/CodeActAgent/o3-mini_maxiter_50_N_v0.25.0-no-hint-run_*/output.pyperf.jsonl"
)

In [4]:
report = load_json(eval_report)

# %% Load traces and create a mapping from instance_id to trace
import pandas as pd
from tqdm import tqdm


def load_traces_to_df(trace_files):
    traces_data = []

    for trace_file in tqdm(trace_files, desc="Loading trace files"):
        run_id = trace_file.parent.name  # Extract run ID from path

        with open(trace_file, "r") as f:
            for line in f:
                try:
                    trace = json.loads(line)
                    if "instance_id" in trace:
                        trace["run_id"] = run_id
                        traces_data.append(trace)
                except json.JSONDecodeError:
                    continue

    return pd.DataFrame(traces_data)


# Load all traces into a dataframe
traces_df = load_traces_to_df(traces)

# Print info about the dataframe
print(f"Loaded {len(traces_df)} trace entries")
print(f"Unique instance IDs: {traces_df['instance_id'].nunique()}")
print(f"Unique run IDs: {traces_df['run_id'].nunique()}")
traces_df.head()

Loading trace files: 100%|██████████| 10/10 [00:03<00:00,  2.82it/s]

Loaded 1135 trace entries
Unique instance IDs: 122
Unique run IDs: 10


,instance_id,test_result,instruction,metadata,history,metrics,error,instance,run_id
0,numpy__numpy-6d77c59,{'git_patch': 'diff --git a/numpy/ma/core.py b...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-11T21:11:20.3...","{'accumulated_cost': 0.27247770000000004, 'cos...",None,"{'instance_id': 'numpy__numpy-6d77c59', 'repo'...",o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1
1,numpy__numpy-28706af,{'git_patch': 'diff --git a/numpy/core/umath.p...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-11T21:11:13.7...","{'accumulated_cost': 0.1779734, 'costs': [{'mo...",None,"{'instance_id': 'numpy__numpy-28706af', 'repo'...",o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1
2,numpy__numpy-5f94eb8,{'git_patch': ''},<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-11T21:11:16.6...","{'accumulated_cost': 0.3823204, 'costs': [{'mo...",None,"{'instance_id': 'numpy__numpy-5f94eb8', 'repo'...",o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1
3,numpy__numpy-ba89ef9,{'git_patch': 'diff --git a/numpy/__init__.py ...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-11T21:11:13.7...","{'accumulated_cost': 0.30504539999999997, 'cos...",None,"{'instance_id': 'numpy__numpy-ba89ef9', 'repo'...",o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1
4,numpy__numpy-7ff7ec7,{'git_patch': 'diff --git a/numpy/_core/numeri...,<uploaded_files>\n/workspace/numpy__numpy\n</u...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-11T21:11:14.2...","{'accumulated_cost': 0.36114760000000007, 'cos...",None,"{'instance_id': 'numpy__numpy-7ff7ec7', 'repo'...",o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1


In [5]:
opt_stats = report["opt_stats"]
opt_stats_data = []

for instance_id, stats in opt_stats.items():
    run_id = stats["report_file"].split("/")[-1].rstrip("test.report.json")
    stats_dict = {
        "instance_id": instance_id,
        "run_id": run_id,
        **stats,  # Unpack all stats
    }
    opt_stats_data.append(stats_dict)

opt_stats_df = pd.DataFrame(opt_stats_data)
print(f"Loaded {len(opt_stats_df)} entries from opt_stats")
opt_stats_df.head()

Loaded 54 entries from opt_stats


,instance_id,run_id,opt_perc_base,opt_perc_commit,opt_perc_main,speedup_base,speedup_commit,speedup_main,geomean_speedup_base,geomean_speedup_commit,...,base_mean,base_std,patch_mean,patch_std,commit_mean,commit_std,main_mean,main_std,per_test_means,report_file
0,numpy__numpy-22ab9aa,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1,43.598230,NaN,NaN,1.772994,NaN,NaN,1.716834,NaN,...,0.064722,0.091771,0.036504,0.050818,0.015791,0.021875,0.006658,0.007462,"{'base': [0.1705114, 0.017159, 0.0064961999999...",reports/o3-mini_maxiter_50_N_v0.25.0-no-hint-r...
1,numpy__numpy-68eead8,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1,98.959609,NaN,NaN,96.117682,NaN,NaN,66.239234,NaN,...,2.615849,4.374389,0.027215,0.044416,0.034942,0.058312,0.022364,0.036854,"{'base': [7.6665458, 0.146424, 0.0345776], 'pa...",reports/o3-mini_maxiter_50_N_v0.25.0-no-hint-r...
2,numpy__numpy-728fedc,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1,42.448679,NaN,NaN,1.737580,NaN,NaN,1.640409,NaN,...,0.334165,0.550364,0.192316,0.313377,0.040229,0.063810,0.029171,0.045478,"{'base': [0.9696684, 0.0150982, 0.0177282], 'p...",reports/o3-mini_maxiter_50_N_v0.25.0-no-hint-r...
3,numpy__numpy-794f474,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1,41.686105,NaN,NaN,1.714857,NaN,NaN,1.714857,NaN,...,0.032152,0.000000,0.018749,0.000000,0.014974,0.000000,0.014446,0.000000,"{'base': [0.0321522], 'patch': [0.0187492], 'c...",reports/o3-mini_maxiter_50_N_v0.25.0-no-hint-r...
4,numpy__numpy-83c780d,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_7,79.746866,NaN,NaN,4.937507,NaN,NaN,2.457928,NaN,...,0.179288,0.220919,0.036312,0.008614,0.031195,0.037065,0.026222,0.027668,"{'base': [0.33550179999999996, 0.0230748], 'pa...",reports/o3-mini_maxiter_50_N_v0.25.0-no-hint-r...


In [6]:
# %% Join the dataframes
# First, prepare the traces dataframe - we want one row per instance_id and run_id
# If there are multiple entries per instance, we'll keep the last one
traces_df_last = traces_df.sort_values(["instance_id", "run_id"]).drop_duplicates(
    ["instance_id", "run_id"], keep="last"
)

# Merge with opt_stats
merged_df = pd.merge(
    traces_df_last,
    opt_stats_df,
    on=["instance_id", "run_id"],
    how="outer",
    suffixes=("_trace", "_stats"),
)

# Check merge results
print(f"Merged dataframe has {len(merged_df)} rows")
print(
    f"Number of instances with both trace and stats: {merged_df.dropna(subset=['instance_id', 'run_id']).shape[0]}"
)

# Display the merged dataframe
merged_df.head()

Merged dataframe has 1135 rows
Number of instances with both trace and stats: 1135


,instance_id,test_result,instruction,metadata,history,metrics,error,instance,run_id,opt_perc_base,...,base_mean,base_std,patch_mean,patch_std,commit_mean,commit_std,main_mean,main_std,per_test_means,report_file
0,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-20T19:32:30.1...","{'accumulated_cost': 0.5058361, 'costs': [{'mo...",None,{'instance_id': 'huggingface__datasets-2878019...,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_1,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-20T20:02:10.9...","{'accumulated_cost': 0.36784110000000003, 'cos...",None,{'instance_id': 'huggingface__datasets-2878019...,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_2,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-20T20:30:02.4...","{'accumulated_cost': 0.35861760000000004, 'cos...",None,{'instance_id': 'huggingface__datasets-2878019...,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_3,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-20T20:50:44.7...","{'accumulated_cost': 0.18435449999999998, 'cos...",None,{'instance_id': 'huggingface__datasets-2878019...,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_4,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,huggingface__datasets-2878019,{'git_patch': 'diff --git a/src/datasets/build...,<uploaded_files>\n/workspace/huggingface__data...,"{'agent_class': 'CodeActAgent', 'llm_config': ...","[{'id': 0, 'timestamp': '2025-03-20T21:16:44.1...","{'accumulated_cost': 0.5410438, 'costs': [{'mo...",None,{'instance_id': 'huggingface__datasets-2878019...,o3-mini_maxiter_50_N_v0.25.0-no-hint-run_5,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
